In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


In [ ]:
from pathlib import Path


In [ ]:
train=pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv")
test=pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv")

train.head()


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [ ]:
print("Train columns:", train.columns.tolist())
print("Test columns :", test.columns.tolist())

# test.dtypes


Train columns: ['id', 'annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade', 'loan_paid_back']
Test columns : ['id', 'annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']


In [ ]:
# Fetching the variety of inputs in the dataset

for column in train.columns:
    if train[column].dtypes=='O':
        print(f"Column name: {column}")
        print(train[column].unique())
        print("="*50)


Column name: gender
['Female' 'Male' 'Other']
Column name: marital_status
['Single' 'Married' 'Divorced' 'Widowed']
Column name: education_level
['High School' "Master's" "Bachelor's" 'PhD' 'Other']
Column name: employment_status
['Self-employed' 'Employed' 'Unemployed' 'Retired' 'Student']
Column name: loan_purpose
['Other' 'Debt consolidation' 'Home' 'Education' 'Vacation' 'Car'
 'Medical' 'Business']
Column name: grade_subgrade
['C3' 'D3' 'C5' 'F1' 'D1' 'D5' 'C2' 'C1' 'F5' 'D4' 'C4' 'D2' 'E5' 'B1'
 'B2' 'F4' 'A4' 'E1' 'F2' 'B4' 'E4' 'B3' 'E3' 'B5' 'E2' 'F3' 'A5' 'A3'
 'A1' 'A2']


# Perorming EDA now

In [ ]:
X=train.drop(columns=["id","loan_paid_back"])
y=train["loan_paid_back"].astype(int)

X_test=test.drop(columns=["id"])


One-Hot-Encoding

In [ ]:
# This is XGBoost firendly
X=pd.get_dummies(X,drop_first=False)
X_test=pd.get_dummies(X_test,drop_first=False)

# Align COlumns
X_test=X_test.reindex(columns=X.columns,fill_value=0)
# This is done to ensure that say in X_train some column has
# B as an output but in test dataset that column never has B as an output 


print("Processed shapes: X",X.shape," X_test:  ",X_test.shape,"y: ",y.shape)


Processed shapes: X (593994, 60)  X_test:   (254569, 60) y:  (593994,)


In [ ]:
proc_dir=Path('/kaggle/working/processed')
proc_dir.mkdir(exist_ok=True)
X.to_parquet(proc_dir/'X.parquet',index=False)
X_test.to_parquet(proc_dir/'X_.parquet',index=False)
y.to_frame().to_parquet(proc_dir/'y.parquet',index=False)

print("Preprocessing Finished")


# Hyper-parameter tuning

OPTUNA- Goated for hypeparameter tuning

 so far we did 20 trials in 

```
study.optimize(objective,n_trials=20)
```
so total trials was 20 to further increase from 20--->40 rather than re-starting the kernel and setting from scratch we can rather just re run the cell which will start from trial 21-40 that's why optuna is good


In [ ]:
import optuna
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 800, 2000),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 4.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),

        # NEW FIXES:
        "tree_method": "hist",
        "device": "cuda",
        "sampling_method": "uniform",  # <-- IMPORTANT for stability
        "eval_metric": "auc"
    }

    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict_proba(X_val)[:, 1]

        # If any NaNs occur → skip trial
        if np.isnan(preds).any():
            return 1  # return worst possible value

        scores.append(roc_auc_score(y_val, preds))

    return 1 - np.mean(scores)


study=optuna.create_study(direction="minimize")
study.optimize(objective,n_trials=20)

print("Best Params:",study.best_params)
print("Best CV AUC",1-study.best_value)


In [ ]:
study.optimize(objective,n_trials=25)

print("Best Params:",study.best_params)
print("Best CV AUC",1-study.best_value)


## Training XG-Boost Model based on the best hyperparameters it receieved (P1)

FINAL GOAL IS TO ENSEMBLE THE MODELS: XGB+LGM+CATBOOST

In [ ]:
xgb_params = study.best_params.copy()
xgb_params.update({
    "tree_method":"hist",
    "device":"cuda",
    "eval_metric":"auc",
    "verbosity":0
})

print(f"Final XGBoost Params:",xgb_params)

# for storing predictions
oof_xgb=np.zeros(len(X))
test_preds_xgb=np.zeros(len(X_test))
kf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

for fold, (train_idx,valid_idx) in enumerate(kf.split(X,y)):
    print(f"\n Fold {fold+1}/5 training")
    X_train,X_valid=X.iloc[train_idx],X.iloc[valid_idx]
    y_train,y_valid=y.iloc[train_idx],y.iloc[valid_idx]

    model=xgb.XGBClassifier(
        **xgb_params,
        early_stopping_rounds=100)

    model.fit(
        X_train,y_train,
        eval_set=[(X_valid,y_valid)],
        verbose=False,
    )
        # Save OOF predictions
    oof_xgb[valid_idx] = model.predict_proba(X_valid)[:, 1]

    # Save test predictions (averaged)
    test_preds_xgb += model.predict_proba(X_test)[:, 1] / kf.n_splits


xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f"\n Final XGBoost OOF AUC: {xgb_oof_auc:.6f}")


## LightGBM Code

In [ ]:
print(study.best_params)


In [ ]:
!pip install lightgbm --quiet


First we will get a baseline lightGBM parameter and then further fine-tune it via <h1>OPTUNA</h1>

In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# -----------------------------
# LightGBM Hyperparameters
# -----------------------------
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 2,
    "lambda_l1": 1.0,
    "lambda_l2": 1.0,
    "min_child_samples": 40,
    "device_type": "gpu",
    "verbose": -1,
}

print("LightGBM Params:", lgb_params)

# -----------------------------
# Arrays for Storage
# -----------------------------
oof_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# 5-Fold Training using callbacks
# -----------------------------
for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"\n LightGBM Fold {fold+1}/5...")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)

    model = lgb.train(
        lgb_params,
        dtrain,
        valid_sets=[dvalid],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
        ]
    )

    # OOF predictions
    oof_lgb[valid_idx] = model.predict(X_valid, num_iteration=model.best_iteration)

    # Test predictions
    test_preds_lgb += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

# -----------------------------
# OOF Score
# -----------------------------
lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f"\n LightGBM OOF AUC: {lgb_oof_auc:.6f}")

# -----------------------------
# Save Outputs
# -----------------------------
import joblib
joblib.dump(model, "lgb_final_model.pkl")
np.save("oof_lgb.npy", oof_lgb)
np.save("test_preds_lgb.npy", test_preds_lgb)

print("\n Saved: lgb_final_model.pkl, oof_lgb.npy, test_preds_lgb.npy")


# Adding CATBOOST Now

In [ ]:
from catboost import CatBoostClassifier
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import joblib

# Original raw data
X_cat = train.drop(columns=['id','loan_paid_back']).copy()
y_cat = train['loan_paid_back'].astype(int).copy()
X_test_cat = test.drop(columns=['id']).copy()

# Categorical columns
cat_cols = ['gender','marital_status','education_level','employment_status','loan_purpose','grade_subgrade']

# Ensure correct dtype
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

# Convert categorical column names → column indices
cat_indices = [X_cat.columns.get_loc(c) for c in cat_cols]

print("Using categorical indices:", cat_indices)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_cat = np.zeros(len(X_cat))
test_preds_cat = np.zeros(len(X_test_cat))

# CatBoost parameters
cat_params = {
    "iterations": 3000,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 5,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "task_type": "GPU",
    "verbose": False,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cat, y_cat)):
    print(f"\n Corrected CatBoost Fold {fold+1}/5...")
    
    X_tr, X_val = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
    y_tr, y_val = y_cat.iloc[tr_idx], y_cat.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_indices,
        early_stopping_rounds=150,
        verbose=False,
    )

    oof_cat[val_idx] = model.predict_proba(X_val)[:,1]
    test_preds_cat += model.predict_proba(X_test_cat)[:,1] / kf.n_splits

# Score
cat_oof_auc = roc_auc_score(y_cat, oof_cat)
print(f"\n Corrected CatBoost OOF AUC: {cat_oof_auc:.6f}")

joblib.dump(model, "cat_final_model.pkl")
np.save("oof_cat.npy", oof_cat)
np.save("test_preds_cat.npy", test_preds_cat)


In [ ]:
print(model.get_params())

import numpy as np
import joblib

np.save("oof_xgb.npy", oof_xgb)
np.save("test_preds_xgb.npy", test_preds_xgb)

joblib.dump(model, "xgb_final_model.pkl")

print("Saved oof_xgb.npy, test_preds_xgb.npy, xgb_final_model.pkl")


## Best Ensembel as of now XGB with LGBM

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Load OOF and test preds
oof_xgb = np.load("oof_xgb.npy")
oof_lgb = np.load("oof_lgb.npy")

test_xgb = np.load("test_preds_xgb.npy")
test_lgb = np.load("test_preds_lgb.npy")

best_auc = 0
best_w = 0

weights = np.linspace(0, 1, 101)  # 0.00, 0.01, ..., 1.00

for w in weights:
    blend = w * oof_xgb + (1 - w) * oof_lgb
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc = auc
        best_w = w

print(f"\n Best Ensemble Weight:")
print(f"  w_xgb = {best_w:.2f}")
print(f"  w_lgb = {1 - best_w:.2f}")
print(f"\n Best OOF AUC = {best_auc:.6f}")

# Blend test predictions with the best weight
test_blend = best_w * test_xgb + (1 - best_w) * test_lgb

# Save submission
import pandas as pd
submission = pd.DataFrame({
    "id": test["id"],
    "loan_paid_back": test_blend
})
submission.to_csv("submission_ensemble_xgb_lgb.csv", index=False)

print("\n Saved: submission_ensemble_xgb_lgb.csv")


# Tune LightBGM with Optuna

Goal today: tune LightGBM with Optuna, get a stronger LGBM (likely the single biggest win), produce 5-fold OOF/test preds, then do seed averaging + stacking if we want an extra push.

In [ ]:
!pip install optuna --quiet


In [ ]:
import optuna, json, joblib, numpy as np, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings, os
warnings.filterwarnings("ignore")


In [ ]:
DB_PATH = "/kaggle/working/lgbm_optuna.db"
PARAM_PATH = "/kaggle/working/best_lgbm_params.json"
MODEL_PATH = "/kaggle/working/best_lgbm_model.pkl"
LOG_PATH = "/kaggle/working/lgbm_tuning_log.txt"


Performing Hyperparameter tuning

In [ ]:
def objective(trial):
    params = {
        "boosting_type": "gbdt",
        "objective": "binary",
        "metric": "auc",
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 24, 512),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),

        # GPU + silent mode
        "device": "gpu",
        "verbosity": -1,

        # IMPORTANT: Kaggle GPU LightGBM supports early stopping ONLY here
        "early_stopping_round": 100,
         
    }

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for tr_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, y_tr)
        dvalid = lgb.Dataset(X_val, y_val)

        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dvalid],
            num_boost_round=4000
        )

        preds = model.predict(X_val, num_iteration=model.best_iteration)
        aucs.append(roc_auc_score(y_val, preds))

    avg_auc = float(np.mean(aucs))

    with open(LOG_PATH, "a") as f:
        f.write(f"Trial {trial.number} AUC: {avg_auc:.6f} params: {json.dumps(params)}\n")

    return 1 - avg_auc



# Load or create study
if os.path.exists(DB_PATH):
    print("Resuming study...")
    study = optuna.load_study(study_name="lgbm_opt", storage=f"sqlite:///{DB_PATH}")
else:
    print("Starting new study...")
    study = optuna.create_study(study_name="lgbm_opt", direction="minimize",
                                storage=f"sqlite:///{DB_PATH}")

# Run tuning
study.optimize(objective, n_trials=40, show_progress_bar=True)
 
print("\nBest value (1-AUC):", study.best_value)
print("Best params:\n", study.best_params)

# Save params
with open(PARAM_PATH, "w") as f:
    json.dump(study.best_params, f, indent=2)


Resuming study...


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2025-11-28 16:43:01,125] Trial 10 finished with value: 0.08239146948148457 and parameters: {'learning_rate': 0.003136049655077345, 'num_leaves': 331, 'max_depth': 4, 'min_child_samples': 146, 'subsample': 0.764892741297043, 'colsample_bytree': 0.46578217524846943, 'reg_alpha': 4.035160867809491, 'reg_lambda': 2.5499100459982103, 'min_split_gain': 0.14037823643557001}. Best is trial 6 with value: 0.07875343565675763.
[I 2025-11-28 16:43:29,095] Trial 11 finished with value: 0.08528107447513078 and parameters: {'learning_rate': 0.00300499158821864, 'num_leaves': 272, 'max_depth': 9, 'min_child_samples': 118, 'subsample': 0.8201109250654425, 'colsample_bytree': 0.7141232818922643, 'reg_alpha': 2.5704693726778776, 'reg_lambda': 4.315799849420241, 'min_split_gain': 0.017191330737787003}. Best is trial 6 with value: 0.07875343565675763.
[I 2025-11-28 16:52:36,236] Trial 12 finished with value: 0.07682540728776532 and parameters: {'learning_rate': 0.019687703094417152, 'num_leaves': 45, 'm

## Building final model params

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import joblib

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 2,
    "lambda_l1": 1.0,
    "lambda_l2": 1.0,
    "min_child_samples": 40,
    "device_type": "gpu",
    "verbose": -1,
}

oof_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n LightGBM Fold {fold+1}/5...")

    X_train, X_valid = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[val_idx]   # <-- FIXED

    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)

    model = lgb.train(
        lgb_params,
        dtrain,
        valid_sets=[dvalid],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
        ]
    )

    oof_lgb[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration)
    test_preds_lgb += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print("\n Baseline LightGBM OOF AUC:", lgb_oof_auc)

np.save("test_preds_lgb.npy", test_preds_lgb)
np.save("oof_lgb.npy", oof_lgb)





LightGBM Fold 1/5...
LightGBM Fold 2/5...
LightGBM Fold 3/5...
LightGBM Fold 4/5...
LightGBM Fold 5/5...
Baseline LightGBM OOF AUC: 0.9229136683159137


In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [31]:
import xgboost as xgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

xgb_params = {
    'n_estimators': 1927,
    'max_depth': 5,
    'learning_rate': 0.0697328011055955,
    'subsample': 0.742898619395241,
    'colsample_bytree': 0.7014450463153368,
    'gamma': 3.3404085682541633,
    'min_child_weight': 9,
    'reg_alpha': 1.1914876918868824,
    'reg_lambda': 4.549633083437573,
    "tree_method":"hist",
    "device":"cuda",
    "eval_metric":"auc",
    "verbosity":0
}

oof_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold+1}/5...")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)

    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    oof_xgb[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds_xgb += model.predict_proba(X_test)[:, 1] / 5

np.save("test_preds_xgb.npy", test_preds_xgb)
np.save("oof_xgb.npy", oof_xgb)

print("XGB OOF:", roc_auc_score(y, oof_xgb))


Fold 1/5...
Fold 2/5...
Fold 3/5...
Fold 4/5...
Fold 5/5...
XGB OOF: 0.9216308777588275
